In [1]:
import shutil
import os

# Clean up all redundant folders and files from previous runs to ensure a 100% fresh start
dirs_to_remove = [
    '/kaggle/working/Final_Thesis_Dataset', 
    '/kaggle/working/Thesis_Data', 
    '/kaggle/working/tmp', 
    '/kaggle/working/tmp_base',
    '/kaggle/working/preprocessed_rsi_5class',
    '/kaggle/working/preprocessed_multimodal_5class'
]

for d in dirs_to_remove:
    if os.path.exists(d):
        shutil.rmtree(d)

# Remove old zip to prevent confusion
if os.path.exists('/kaggle/working/Thesis_Data.zip'):
    os.remove('/kaggle/working/Thesis_Data.zip')

In [2]:
# 1. Install MONAI
!pip install monai[nibabel] -q

# 2. Setup imports
import os
import shutil
import zipfile
import numpy as np
from pathlib import Path
from tqdm import tqdm
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, 
    Spacingd, CropForegroundd, NormalizeIntensityd
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 6.9 MB/s eta 0:00:00


In [3]:
ZIP_PATH = '/kaggle/input/datasets/iftekhar994/ucsd-brain-tumor-scans-v2/brain_data.bin'
RSI_DIR = Path('/kaggle/working/preprocessed_rsi_5class')
BASE_DIR = Path('/kaggle/working/preprocessed_multimodal_5class')
RSI_DIR.mkdir(exist_ok=True)
BASE_DIR.mkdir(exist_ok=True)

# Define configurations
configs = [
    {'out': RSI_DIR, 'keys': ['t1', 't1ce', 't2', 'flair', 'rsi', 'adc'], 'map': {'t1': 'T1pre', 't1ce': 'T1post', 't2': 'T2', 'flair': 'FLAIR', 'rsi': 'RSI_Cell', 'adc': 'ADC_vendor', 'label': 'BraTS_tumor_seg'}},
    {'out': BASE_DIR, 'keys': ['t1', 't1ce', 't2', 'flair'], 'map': {'t1': 'T1pre', 't1ce': 'T1post', 't2': 'T2', 'flair': 'FLAIR', 'label': 'BraTS_tumor_seg'}}
]

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()
    p_list = sorted(list(set([f.split('_BraTS_tumor_seg')[0].split('/')[-1] for f in all_files if 'BraTS_tumor_seg' in f])))

    # Loop over all 184 patients dynamically
    for pid in tqdm(p_list, desc='Processing Datasets'):
        for cfg in configs:
            out_path = cfg['out'] / f'{pid}.npz'
            if out_path.exists(): continue
            
            p_dir = Path(f'/kaggle/working/tmp_{cfg["out"].name}/{pid}')
            p_dir.mkdir(parents=True, exist_ok=True)
            
            try:
                p_dict = {}
                for key, suffix in cfg['map'].items():
                    target = f"{pid}_{suffix}.nii.gz"
                    match = [f for f in all_files if f.endswith(target)]
                    if not match: continue
                    dest = p_dir / target
                    with z.open(match[0]) as src, open(dest, 'wb') as dst: dst.write(src.read())
                    p_dict[key] = str(dest)
                
                # --- DYNAMIC INTERPOLATION FIX ---
                current_keys = list(p_dict.keys())
                interp_modes = ['bilinear' if k != 'label' else 'nearest' for k in current_keys]
                
                # Preprocess Pipeline
                pp = Compose([
                    LoadImaged(keys=current_keys), 
                    EnsureChannelFirstd(keys=current_keys), 
                    Orientationd(keys=current_keys, axcodes='RAS'), 
                    Spacingd(keys=current_keys, pixdim=(1.5, 1.5, 1.5), mode=interp_modes), # Safe interpolation applied
                    CropForegroundd(keys=current_keys, source_key='flair', select_fn=lambda x: x > 0, margin=4), 
                    NormalizeIntensityd(keys=cfg['keys'], nonzero=True, channel_wise=True)
                ])
                
                data = pp(p_dict)
                img = np.concatenate([data[k].numpy() for k in cfg['keys']], axis=0).astype(np.float32)
                lbl = data['label'].numpy().astype(np.uint8)
                np.savez_compressed(out_path, image=img, label=lbl)
            finally:
                shutil.rmtree(p_dir, ignore_errors=True)
                
print("\nAll 184 files processed cleanly with zero boundary artifacts!")

Processing Datasets:   0%|          | 0/184 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)
Processing Datasets: 100%|██████████| 184/184 [37:31<00:00, 12.24s/it]


All 184 files processed cleanly with zero boundary artifacts!


In [4]:
import os
import shutil

# Staging
if os.path.exists('/kaggle/working/Final_Thesis_Dataset'): shutil.rmtree('/kaggle/working/Final_Thesis_Dataset')
os.makedirs('/kaggle/working/Final_Thesis_Dataset')

# Copy to staging
shutil.copytree(RSI_DIR, '/kaggle/working/Final_Thesis_Dataset/preprocessed_rsi_5class')
shutil.copytree(BASE_DIR, '/kaggle/working/Final_Thesis_Dataset/preprocessed_multimodal_5class')

# Archive
if os.path.exists('/kaggle/working/Thesis_Data.zip'): os.remove('/kaggle/working/Thesis_Data.zip')
shutil.make_archive('/kaggle/working/Thesis_Data', 'zip', '/kaggle/working/Final_Thesis_Dataset')

print("Archive rebuilt successfully. Thesis_Data.zip is ready.")

Archive rebuilt successfully. Thesis_Data.zip is ready.
